In [15]:
import os
os.environ["PYTHONHASHSEED"] = "1234"  
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8" 

import random
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoderLayer, TransformerEncoder
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, precision_recall_curve
from positional_encodings.torch_encodings import PositionalEncoding1D
from tqdm.notebook import tqdm

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.use_deterministic_algorithms(True)
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### 파라미터 설정

In [16]:
BATCH_SIZE = 256
DATA_DIR = "./yahoo/A1Benchmark/"
WEIGHT_DECAY = 1e-3
NUM_EPOCHS = 57

# 입력 시계열 길이 및 패치 크기 설정
global_seq_len = 300
patch_size = 60

In [17]:
with open('yahoo_a1_splits.json', 'r') as f: 
	best_split = json.load(f)

#### 데이터

In [18]:
def create_sliding_windows(arr, window_size, step):
    windows = [arr[i: i + window_size] for i in range(0, len(arr) - window_size + 1, step)]
    return np.array(windows)

def process_file(df, window_size, step, scaler=None):
    values = df["value"].values.astype(np.float32)
    labels = df["is_anomaly"].values.astype(np.float32)

    if scaler == None:
        scaler = RobustScaler()
    values = scaler.fit_transform(values.reshape(-1,1)).reshape(values.shape)

    X_windows = create_sliding_windows(values, window_size, step)
    y_windows = create_sliding_windows(labels, window_size, step)
    
    return X_windows, y_windows, scaler

X_train, y_train = [], []
X_val, y_val = [], []
X_test, y_test = [], []
scaler_dict = {}
step = 1

for split in best_split:
    for fp in best_split[split]:
        df = pd.read_csv(fp)
        X_win, y_win, scaler = process_file(df, global_seq_len, step)
        if split == "train":
            X_train.append(X_win)
            y_train.append(y_win)
        elif split == "val":
            X_val.append(X_win)
            y_val.append(y_win)
        elif split == "test":
            X_test.append(X_win)
            y_test.append(y_win)
        scaler_dict[fp.split('/')[-1]] = scaler

X_train, y_train_ts = np.concatenate(X_train, axis=0), np.concatenate(y_train, axis=0)
X_val, y_val_ts = np.concatenate(X_val, axis=0), np.concatenate(y_val, axis=0)
X_test, y_test_ts = np.concatenate(X_test, axis=0), np.concatenate(y_test, axis=0)
print(f"Total train windows: {X_train.shape[0]}, Total val windows: {X_val.shape[0]}, Total test windows: {X_test.shape[0]}")

Total train windows: 44822, Total val windows: 14814, Total test windows: 15197


In [19]:
def create_patch(X, y, patch_size):
    X_patch = X.reshape(X.shape[0],-1,patch_size)
    y_patch = y.reshape(y.shape[0],-1,patch_size).sum(axis=2)>0
    y_patch = y_patch.astype(float)
    return X_patch, y_patch

X_train, y_train_patch = create_patch(X_train, y_train_ts, patch_size)
X_val, y_val_patch = create_patch(X_val, y_val_ts, patch_size)
X_test, y_test_patch = create_patch(X_test, y_test_ts, patch_size)

In [20]:
def compute_pos_weight(y):
    y = y.flatten()
    num_pos = y.sum()
    num_neg = len(y) - num_pos
    pos_weight = num_neg / num_pos
    return torch.tensor([pos_weight], dtype=torch.float)

pos_weight_ts = compute_pos_weight(y_train_ts)
pos_weight_patch = compute_pos_weight(y_train_patch)
print(f"pos_weight_ts: {pos_weight_ts}, pos_weight_patch: {pos_weight_patch}")

pos_weight_ts: tensor([73.8272]), pos_weight_patch: tensor([11.4894])


In [21]:
g = torch.Generator()
g.manual_seed(SEED)

In [22]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y_ts, y_patch):
        self.X = X
        self.y_ts = y_ts
        self.y_patch = y_patch

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (torch.tensor(self.X[idx], dtype=torch.float32),
                torch.tensor(self.y_ts[idx], dtype=torch.float32),
                torch.tensor(self.y_patch[idx], dtype=torch.float32))
        
train_dataset = TimeSeriesDataset(X_train, y_train_ts, y_train_patch)
val_dataset = TimeSeriesDataset(X_val, y_val_ts, y_val_patch)
test_dataset = TimeSeriesDataset(X_test, y_test_ts, y_test_patch)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True,generator=g,num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True, num_workers=0)

#### 모델 설계

In [23]:
class FOCUS(nn.Module):
    def __init__(self,
                 seq_len: int,
                 patch_size: int,
                 emb_dim: int = 256,
                 num_heads: int = 8,
                 num_layers: int = 6,
                 dropout: float = 0.3):
        super().__init__()
        assert seq_len % patch_size == 0, "Sequence length must be divisible by patch size"
        self.seq_len = seq_len
        self.patch_size = patch_size
        self.num_patches = seq_len // patch_size
        self.emb_dim = emb_dim

        # patch embedding: (B,1,L) -> (B,emb_dim,M) where M=num_patches
        self.patch_embed = nn.Conv1d(1, self.emb_dim, self.patch_size, self.patch_size)
        self.norm = nn.LayerNorm(self.emb_dim)

        # CLS 제거: 위치 임베딩만 패치 토큰 길이에 맞게 더해줌
        self.pos_embed = PositionalEncoding1D(self.emb_dim)

        encoder_layer = TransformerEncoderLayer(
            d_model=self.emb_dim,
            nhead=num_heads,
            dim_feedforward=self.emb_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = TransformerEncoder(encoder_layer, num_layers=num_layers)

        # (A) Patch-level head: CLS 대신 mean-pooled patch 토큰으로 요약
        self.patch_pool = nn.AdaptiveAvgPool1d(1)  # (B, D, M) -> (B, D, 1)
        self.cls_head = nn.Sequential(
            nn.LayerNorm(self.emb_dim),
            nn.Linear(self.emb_dim, self.num_patches),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )

        # (B) Timestamp-level head: 각 패치 토큰을 patch_size 길이로 복원
        self.time_head = nn.Sequential(
            nn.LayerNorm(self.emb_dim),
            nn.Linear(self.emb_dim, self.patch_size),
            nn.GELU(),
            nn.Dropout(p=dropout),
        )

    def forward(self, x):
        """
        x: (B, seq_len)
        returns:
          y_time: (B, seq_len)
          y_patch: (B, num_patches)
        """
        B = x.size(0)
        # (B, L) -> (B,1,L)
        x = x.view(B, 1, -1)
        # (B,1,L) -> (B, D, M)
        x = self.patch_embed(x)
        # (B, D, M) -> (B, M, D)
        x = x.transpose(1, 2)
        x = self.norm(x)

        # CLS 없이 positional encoding을 패치 토큰에만 적용
        x = x + self.pos_embed(x)     # (B, M, D)

        # Transformer 인코더(입력 길이 = num_patches)
        z = self.transformer(x)       # (B, M, D)

        # ---- (A) Patch-level prediction: mean-pool over patches ----
        # z: (B, M, D) -> (B, D, M) -> pool -> (B, D, 1) -> squeeze -> (B, D)
        z_t = z.transpose(1, 2)
        pooled = self.patch_pool(z_t).squeeze(-1)
        y_patch = self.cls_head(pooled)  # (B, num_patches)

        # ---- (B) Timestamp-level prediction: per-patch -> per-timestep ----
        # z: (B, M, D) -> time_head -> (B, M, P) -> reshape -> (B, L)
        y_time_patch = self.time_head(z)         # (B, num_patches, patch_size)
        y_time = y_time_patch.reshape(B, self.seq_len)

        return y_time, y_patch

In [24]:
class UncertaintyLoss(nn.Module):
    def __init__(self, num_tasks=2, clamp_min=0.1, clamp_max=5.0):
        super().__init__()
        self.log_vars = nn.Parameter(torch.zeros(num_tasks))
        self.clamp_min = clamp_min
        self.clamp_max = clamp_max

    def forward(self, *losses):
        total = 0.0
        with torch.no_grad():
            min_val = torch.log(torch.tensor(self.clamp_min**2)).to(self.log_vars.device)
            max_val = torch.log(torch.tensor(self.clamp_max**2)).to(self.log_vars.device)
            self.log_vars.data.clamp_(min=min_val, max=max_val)
        for i, loss in enumerate(losses):
            precision = torch.exp(-self.log_vars[i])
            total += 0.5 * precision * loss + 0.5 * self.log_vars[i]
        return total

#### 학습

In [25]:
class EarlyStopping:
    def __init__(self, patience=15, verbose=True):
        self.patience = patience
        self.verbose = verbose
        self.best_ts = None
        self.best_patch = None
        self.ts_counter = 0
        self.patch_counter = 0
        self.early_stop = False
    
    def __call__(self, val_ts_loss, val_patch_loss):
        if self.best_ts is None:
            self.best_ts = val_ts_loss
            self.ts_counter = 0
        else:
            # improvement_ts = (self.best_ts-val_ts_loss) / self.best_ts
            # if improvement_ts > 0.01:
            if self.best_ts > val_ts_loss:
                self.best_ts = val_ts_loss
                self.ts_counter = 0
            else:
                self.ts_counter += 1

        if self.best_patch is None:
            self.best_patch = val_patch_loss
            self.patch_counter = 0
        else:
            # improvement_patch = (self.best_patch-val_patch_loss) / self.best_patch
            # if improvement_patch > 0.01:
            if self.best_patch > val_patch_loss:
                self.best_patch = val_patch_loss
                self.patch_counter = 0
            else:
                self.patch_counter += 1

        if self.verbose:
            # print(f"TS counter: {min(self.ts_counter,self.patience)}/{self.patience} | Patch counter: {min(self.patch_counter,self.patience)}/{self.patience}")
            pass
        
        if self.ts_counter >= self.patience and self.patch_counter >= self.patience:
            self.early_stop = True

In [26]:
def train_epoch(model, dataloader, optimizer, device,loss_fn_ts, loss_fn_patch,uncertainty_loss=None,scheduler=None):
    model.train()
    total_loss = 0
    total_loss_ts = 0
    total_loss_patch = 0
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for X, y_ts, y_patch in progress_bar:
        X = X.to(device)
        y_ts = y_ts.to(device)
        y_patch = y_patch.to(device)
        
        optimizer.zero_grad()
        y_pred_ts, y_pred_patch = model(X)
        loss_ts = loss_fn_ts(y_pred_ts.flatten(), y_ts.flatten())
        loss_patch = loss_fn_patch(y_pred_patch.flatten(), y_patch.flatten())
        if uncertainty_loss is not None:
            loss = uncertainty_loss(loss_ts, loss_patch)
        else:
            loss = loss_ts + loss_patch
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        total_loss_ts += loss_ts.item()
        total_loss_patch += loss_patch.item()
    
    avg_loss_ts = total_loss_ts / len(dataloader)
    avg_loss_patch = total_loss_patch / len(dataloader)
    # print(f"[Train] TS Loss: {avg_loss_ts:.4f}, Patch Loss: {avg_loss_patch:.4f}")
    return total_loss / len(dataloader), avg_loss_ts, avg_loss_patch

def evaluate(model, dataloader, device, loss_fn_ts, loss_fn_patch,uncertainty_loss=None):
    model.eval()
    total_loss = 0
    total_loss_ts = 0
    total_loss_patch = 0
    all_ts_preds, all_ts_labels = [], []
    all_patch_preds, all_patch_labels = [], []

    with torch.no_grad():
        for X, y_ts, y_patch in tqdm(dataloader, desc="Evaluating", leave=False):
            X = X.to(device)
            y_ts = y_ts.to(device)
            y_patch = y_patch.to(device)

            y_pred_ts, y_pred_patch = model(X)

            loss_ts = loss_fn_ts(y_pred_ts.flatten(), y_ts.flatten())
            loss_patch = loss_fn_patch(y_pred_patch.flatten(), y_patch.flatten())
            if uncertainty_loss is not None:
                loss = uncertainty_loss(loss_ts, loss_patch)
            else:
                loss = loss_ts + loss_patch
            total_loss += loss.item()
            total_loss_ts += loss_ts.item()
            total_loss_patch += loss_patch.item()
            
            all_ts_preds.append(torch.sigmoid(y_pred_ts).flatten().cpu().detach().numpy())
            all_ts_labels.append(y_ts.flatten().cpu().detach().numpy())
            all_patch_preds.append(torch.sigmoid(y_pred_patch).flatten().cpu().detach().numpy())
            all_patch_labels.append(y_patch.flatten().cpu().detach().numpy())

    avg_loss = total_loss / len(dataloader)
    avg_loss_ts = total_loss_ts / len(dataloader)
    avg_loss_patch = total_loss_patch / len(dataloader)
    # print(f"[Eval] TS Loss: {avg_loss_ts:.4f}, Patch Loss: {avg_loss_patch:.4f}")
    all_ts_preds = np.concatenate(all_ts_preds, axis=0)
    all_ts_labels = np.concatenate(all_ts_labels, axis=0)
    all_patch_preds = np.concatenate(all_patch_preds, axis=0)
    all_patch_labels = np.concatenate(all_patch_labels, axis=0)

    return avg_loss, all_ts_preds, all_ts_labels, all_patch_preds, all_patch_labels, avg_loss_ts, avg_loss_patch

def compute_metrics(y_true, y_pred, threshold=0.5):
    preds = (y_pred >= threshold).astype(int)
    precision = precision_score(y_true, preds, zero_division=0)
    recall = recall_score(y_true, preds, zero_division=0)
    f1 = f1_score(y_true, preds, zero_division=0)
    auroc = roc_auc_score(y_true, y_pred)
    return precision, recall, f1, auroc

def threshold_sweep(y_true, y_pred, metric="f1"):
    precision, recall, thresholds = precision_recall_curve(y_true, y_pred)
    precision, recall = precision[:-1], recall[:-1]
    if metric == "precision":
        scores = precision
    elif metric == "recall":
        scores = recall
    else:
        scores = 2 * precision * recall / (precision + recall + 1e-8)
    idx = np.argmax(scores)
    return thresholds[idx], scores[idx]

In [27]:
model = FOCUS(seq_len=global_seq_len, patch_size=patch_size, dropout=0.3).to(DEVICE)

uncertainty_loss = UncertaintyLoss().to(DEVICE)

loss_fn_ts = nn.BCEWithLogitsLoss(pos_weight=pos_weight_ts).to(DEVICE)
loss_fn_patch = nn.BCEWithLogitsLoss(pos_weight=pos_weight_patch).to(DEVICE)

optimizer = optim.AdamW([
    {'params': model.parameters(), 'weight_decay': 4.5e-6},
    {'params': uncertainty_loss.parameters(), 'weight_decay': 0.0},
], lr=3e-4)

early_stopping = EarlyStopping(patience=15, verbose=True)

In [28]:
train_losses_ts, train_losses_patch, val_losses_ts, val_losses_patch = [], [], [], []
best_val_metric = best_thresh_ts = best_thresh_patch = best_epoch = 0
best_model_state = None

for epoch in range(1, NUM_EPOCHS+1):
    train_loss, train_loss_ts, train_loss_patch = train_epoch(model, train_loader, optimizer, DEVICE, loss_fn_ts, loss_fn_patch, uncertainty_loss)
    val_loss, val_ts_preds, val_ts_labels, val_patch_preds, val_patch_labels, val_loss_ts, val_loss_patch = evaluate(model, val_loader, DEVICE, loss_fn_ts, loss_fn_patch, uncertainty_loss)
    train_losses_ts.append(train_loss_ts)
    train_losses_patch.append(train_loss_patch)
    val_losses_ts.append(val_loss_ts)
    val_losses_patch.append(val_loss_patch)
    print(f"Epoch {epoch}/{NUM_EPOCHS}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
    
    thresh_ts, f1_ts = threshold_sweep(val_ts_labels.flatten(), val_ts_preds.flatten(), metric="f1")
    thresh_patch, f1_patch = threshold_sweep(val_patch_labels.flatten(), val_patch_preds.flatten(), metric="f1")

    composite = (f1_ts + f1_patch) / 2
    early_stopping(val_loss_ts, val_loss_patch)
    if best_val_metric < composite:
        best_val_metric = composite
        best_model_state = model.state_dict()
        best_thresh_ts = thresh_ts
        best_thresh_patch = thresh_patch
        best_epoch = epoch
    
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break
    # scheduler.step()
if best_model_state is not None:
    model.load_state_dict(best_model_state)
# print(f"Best Epoch {best_epoch}")

Training:   0%|          | 0/176 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 1/57: Train Loss = 1.1989, Val Loss = 1.1909


Training:   0%|          | 0/176 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/58 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 30/57: Train Loss = 0.7969, Val Loss = 0.9295


Training:   0%|          | 0/176 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/58 [00:00<?, ?it/s]

Epoch 31/57: Train Loss = 0.7878, Val Loss = 0.9200
Early stopping triggered.


In [29]:
test_loss, ts_preds, ts_labels, patch_preds, patch_labels, test_loss_ts, test_loss_patch = evaluate(model, test_loader, DEVICE, loss_fn_ts, loss_fn_patch, uncertainty_loss)

precision_ts, recall_ts, f1_ts, auroc_ts = compute_metrics(ts_labels.flatten(), ts_preds.flatten(), best_thresh_ts)
precision_patch, recall_patch, f1_patch, auroc_patch = compute_metrics(patch_labels.flatten(), patch_preds.flatten(), best_thresh_patch)

print("=== 최종 Test 평가 (Threshold Sweep 적용) ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Timestamp-level -> 최적 Threshold: {best_thresh_ts:.2f}, Precision: {precision_ts:.4f}, Recall: {recall_ts:.4f}, F1: {f1_ts:.4f}, AUROC: {auroc_ts:.4f}")
print(f"Patch-level     -> 최적 Threshold: {best_thresh_patch:.2f}, Precision: {precision_patch:.4f}, Recall: {recall_patch:.4f}, F1: {f1_patch:.4f}, AUROC: {auroc_patch:.4f}")

Evaluating:   0%|          | 0/60 [00:00<?, ?it/s]

=== 최종 Test 평가 (Threshold Sweep 적용) ===
Test Loss: 0.7354
Timestamp-level -> 최적 Threshold: 0.96, Precision: 0.9000, Recall: 0.6889, F1: 0.7804, AUROC: 0.9691
Patch-level     -> 최적 Threshold: 0.51, Precision: 0.7137, Recall: 0.7980, F1: 0.7535, AUROC: 0.9035
